In [1]:
import threading
import traceback
import solara
import math
import random
import time
import numpy as np
import mesa
from mesa.space import SingleGrid
import matplotlib.pyplot as plt
from matplotlib import animation

In [5]:
class Rietkerk_Model(mesa.Model):
    def __init__(self, n=50, rng=None, plant_rate=0.0,surface_rate=0.0,infiltrated_rate=0.0, plant_loss_rate=0.0, plant_water_uptake=0.0,k1=0.0,c=0.0,plant_diffusion=0.0,infiltrated_diffusion=0.0,infiltrated_water_loss=0.0,alpha=0.0,Wo=0.0,k2=0.0,rainfall=0.0,surface_diffusion=0.0):
        super().__init__(rng=rng)
        self.grid = SingleGrid(n,n, torus=True)
        self.n = n

        for x in range(n):
            for y in range(n):
                biomass = plant_rate + random.uniform(0.0, plant_rate/60)
                surface_water = surface_rate + random.uniform(-0.1, 0.1)
                infiltrated_water = infiltrated_rate + random.uniform(-0.1, 0.1)
                agente = AgenteSavana(model=self, biomass = biomass, surface_water = surface_water, infiltrated_water = infiltrated_water,plant_loss_rate= plant_loss_rate, plant_water_uptake= plant_water_uptake,k1=k1,c=c,plant_diffusion=plant_diffusion,infiltrated_diffusion=infiltrated_diffusion,infiltrated_water_loss=infiltrated_water_loss,alpha=alpha,Wo=Wo,k2=k2,rainfall=rainfall,surface_diffusion=surface_diffusion)
                self.grid.place_agent(agente, (x, y))
    def step(self):
        for agent in self.agents:
            agent.prepare_update()

        for agent in self.agents:
            agent.apply_update()

    def export(self):
        matrix_biomass = np.zeros((self.n, self.n))
        matrix_infiltrated_water = np.zeros((self.n, self.n))
        matrix_surface_water = np.zeros((self.n, self.n))

        for agent, (x, y) in self.grid.coord_iter():
            if agent:
                matrix_biomass[x, y] = agent.biomass
                matrix_infiltrated_water[x, y] = agent.infiltrated_water
                matrix_surface_water[x, y] = agent.surface_water
        return matrix_biomass

In [9]:
class AgenteSavana(mesa.Agent):
    def __init__(self, model, biomass, surface_water, infiltrated_water, plant_loss_rate=0.0, plant_water_uptake=0.0,k1=0.0,c=0.0,plant_diffusion=0.0,infiltrated_diffusion=0.0,infiltrated_water_loss=0.0,alpha=0.0,Wo=0.0,k2=0.0,rainfall=0.0,surface_diffusion=0.0):

        # Plant related parameters
        #plant_loss_rate = 0.45       # Mortalidade das plantas                            (d)
        #plant_water_uptake = 0.1     # Absorção máxima de agua das raizes             (gMax)
        #k1 = 5                       # Constante de meia saturação                       (k1)
        #c = 5                        # Coeficiente de conversão da Água                   (c)
        #plant_diffusion = 0.01       # Coeficiente de difusão das plantas                (Db)

        # Infiltrated water related parameters
        #infiltrated_diffusion = 0.01 # Coeficiente de difusão agua infiltrada            (Dw)
        #infiltrated_water_loss = 0.004 # Perda de água do solo por evaporação e drenagem.  (rw)
        #alpha = 0.4                  # Taxa máxima de infiltração                      (alpha)
        #Wo = 0.2                     # Infiltração no solo Nu
        #k2 = 5.0                     # Constante de saturação da infiltração              (k2)

        # surface water related parameters
        #rainfall = 3.5               # Chuva (precipitação)                                (R)
        #surface_diffusion = 50       # Coeficiente de difusão da agua na superficie       (Do)

        # VERSION 2


        dX = 2

        super().__init__(model)
        self.biomass = biomass
        self.next_biomass = biomass
        self.plant_loss_rate = plant_loss_rate
        self.plant_water_uptake = plant_water_uptake
        self.k1 = k1
        self.c = c
        self.plant_diffusion = plant_diffusion

        self.infiltrated_water = infiltrated_water
        self.next_infiltrated_water = infiltrated_water
        self.infiltrated_diffusion = infiltrated_diffusion
        self.infiltrated_water_loss = infiltrated_water_loss
        self.alpha = alpha
        self.Wo = Wo
        self.k2 = k2

        self.surface_water = surface_water
        self.next_surface_water = surface_water
        self.rainfall = rainfall
        self.surface_diffusion = surface_diffusion

        self.dX = dX
        self.passotempo = 0.01 #0.00025


    def prepare_update(self):
        neighbors = self.model.grid.get_neighbors(
            self.pos, moore=False, include_center=False
        )

        Left_Biomass, Right_Biomass, Up_Biomass, Down_Biomass = self.biomass, self.biomass, self.biomass, self.biomass
        Left_Infiltrated_water, Right_Infiltrated_water, Up_Infiltrated_water, Down_Infiltrated_water = self.infiltrated_water, self.infiltrated_water, self.infiltrated_water, self.infiltrated_water
        Left_surface_water, Right_surface_water, Up_surface_water, Down_surface_water = self.surface_water, self.surface_water, self.surface_water, self.surface_water

        x, y = self.pos
        n = self.model.grid.width
        for v in neighbors:
            vx, vy = v.pos
            dx = (vx - x + n) % n
            if dx > n / 2:
                dx -= n
            dy = (vy - y + n) % n
            if dy > n / 2:
                dy -= n

            if dx == -1 and dy == 0:
                Left_Biomass = v.biomass
                Left_Infiltrated_water = v.infiltrated_water
                Left_surface_water = v.surface_water
            elif dx == 1 and dy == 0:
                Right_Biomass = v.biomass
                Right_Infiltrated_water = v.infiltrated_water
                Right_surface_water = v.surface_water
            elif dx == 0 and dy == 1:
                Up_Biomass = v.biomass
                Up_Infiltrated_water = v.infiltrated_water
                Up_surface_water = v.surface_water
            elif dx == 0 and dy == -1:
                Down_Biomass = v.biomass
                Down_Infiltrated_water = v.infiltrated_water
                Down_surface_water = v.surface_water

        biomass_laplacian = ((Left_Biomass + Right_Biomass - 2 * self.biomass) / self.dX / self.dX + (
                    Up_Biomass + Down_Biomass - 2 * self.biomass) / self.dX / self.dX)

        infiltrated_water_laplacian = ((Left_Infiltrated_water + Right_Infiltrated_water - 2 * self.infiltrated_water) / self.dX / self.dX + (
            Up_Infiltrated_water + Down_Infiltrated_water - 2 * self.infiltrated_water) / self.dX / self.dX)

        surface_water_laplacian = ((Left_surface_water + Right_surface_water - 2 * self.surface_water) / self.dX / self.dX + (
            Up_surface_water + Down_surface_water - 2 * self.surface_water) / self.dX / self.dX)

        self.next_biomass = self.biomass + (self.c * self.plant_water_uptake * (self.infiltrated_water / (self.infiltrated_water + self.k1)) * self.biomass - (self.plant_loss_rate * self.biomass) + self.plant_diffusion * biomass_laplacian) * self.passotempo

        self.next_infiltrated_water = self.infiltrated_water + (self.alpha * self.surface_water * ((self.biomass + self.k2 * self.Wo) / (self.biomass + self.k2)) - self.plant_water_uptake * (self.infiltrated_water / (self.infiltrated_water + self.k1)) * self.biomass - self.infiltrated_water_loss * self.infiltrated_water +  self.infiltrated_diffusion * infiltrated_water_laplacian)  * self.passotempo

        self.next_surface_water = self.surface_water + (self.rainfall - self.alpha * self.surface_water * ((self.biomass + self.k2 * self.Wo) / (self.biomass + self.k2)) + self.surface_diffusion * surface_water_laplacian ) * self.passotempo

    def apply_update(self):
        self.biomass = self.next_biomass
        self.infiltrated_water = self.next_infiltrated_water
        self.surface_water = self.next_surface_water

In [2]:

# Declare reactive variables at the top level. Components using these variables
# will be re-executed when their values change.

# --- Plant related parameters
#plant_loss_rate = 0.15 			# Mortalidade/perda das plantas (d)
#plant_water_uptake = 0.1 		# Absorção máxima de água pelas raízes (gMax)
k1 = 5.0 				# Constante de meia-saturação da absorção (k1)
c = 10.0 				# Coeficiente de conversão de água em biomassa (c)
#plant_diffusion = 0.01 			# Coeficiente de dispersão/difusão vegetal (Db)

# --- Infiltrated water related parameters
#infiltrated_diffusion = 0.01 		# Coeficiente de difusão da água no solo (Dw)
#infiltrated_water_loss = 0.2 		# Taxa de perda por evaporação e drenagem profunda
alpha = 0.4 				# Taxa máxima de infiltração facilitada por vegetação/SOM (alpha)
Wo = 0.1 				# Infiltração em solo nu/degradado (w0)
k2 = 5.0 				# Constante de saturação da infiltração (k2)

# --- Surface water related parameters
#rainfall = 3.66 			# Chuva diária média (R)
#surface_diffusion = 50.0 		# Coeficiente de difusão rápida da água na superfície (Do)


plant_rate = solara.reactive(30)
plant_loss = solara.reactive(15)
plant_water_uptake = solara.reactive(10)
plant_difussion = solara.reactive(1)

infiltrated_diffusion = solara.reactive(1)
infiltrated_water_loss = solara.reactive(20)

surface_rate = solara.reactive(10)
water_surface_diffusion = solara.reactive(50)

infiltrated_rate = solara.reactive(10)
water_infiltrated_loss = solara.reactive(20)
water_infiltrated_diffusion = solara.reactive(1)
rainfall = solara.reactive(3.66)
trigger = solara.reactive(0)

def resetParams():
    plant_rate.value = 30
    surface_rate.value = 10
    infiltrated_rate.value = 10
    plant_loss.value = 15
    plant_water_uptake.value = 10
    plant_difussion.value = 1

    infiltrated_diffusion.value = 1
    infiltrated_water_loss.value = 20

    water_surface_diffusion.value = 50

    water_infiltrated_diffusion.value = 1
    water_infiltrated_loss.value = 20
    rainfall.value = 3.66
    trigger.value = 0

@solara.component
def Page():
    # Calculate word_count within the component to ensure re-execution when reactive variables change.
    with solara.Sidebar():
        with solara.Card("Parâmetros", margin=0, elevation=0):
            with solara.Column(style={"max-width":"20vw"}):
                solara.Text("Biomassa",
                            style={"color": "gray", "border-bottom":"2px solid gray", "font-size": "17px", "width":"100%", "padding-bottom":"4px"})
                solara.Text("Relação de biomassa inicial (%)")
                solara.SliderFloat(label=None, value=plant_rate, min=0, max=100, step=0.5)

                solara.Text("Mortalidade das Plantas (%/dia)")
                solara.SliderFloat(label=None, value=plant_loss, min=0.0, max=100)

                solara.Text("Difussão de biomasa")
                solara.SliderFloat(label=None ,value=plant_difussion, min=0.0, max=10)

                solara.Text("Absorção Máxima de água pelas raízes")
                solara.SliderFloat(label=None, value=plant_water_uptake, min=0.0, max=100, step=0.1)

                # ==================================================================================================================================
                solara.Text("Água na superficies",
                            style={"color": "gray", "border-bottom":"2px solid gray", "font-size": "17px", "width":"100%", "padding-bottom":"4px"})

                solara.Text("Relação de água na superficie inicial (%)")
                solara.SliderFloat(label=None, value=surface_rate, min=0, max=100, step=0.5)

                solara.Text("Precipitação (mm/dia)")
                solara.SliderFloat(label=None, value=rainfall, min=0.0, max=5,step=0.01)

                solara.Text("Difussão água na superficie")
                solara.SliderFloat(label=None, value=water_surface_diffusion, min=0.0, max=100,step=0.01)

                # ==================================================================================================================================
                solara.Text("Água infiltrada",
                            style={"color": "gray", "border-bottom":"2px solid gray", "font-size": "17px", "width":"100%", "padding-bottom":"4px"})
                solara.Text("Relação de água infiltrada inicial (%)")
                solara.SliderFloat(label=None, value=infiltrated_rate, min=0, max=100, step=0.5)

                solara.Text("Perda de água do solo (%/dia)")
                solara.SliderFloat(label=None, value=water_infiltrated_loss, min=0.0, max=75,step=0.01)

                solara.Text("Difussão água infiltrada")
                solara.SliderFloat(label=None, value=water_infiltrated_diffusion, min=0.0, max=50,step=0.01)
                solara.Button("Resetar Parâmetros", on_click=resetParams, color="info")

    with solara.Column():
        with solara.Card("Simulação Rietkerk Model"):

            model = solara.use_memo(
                lambda: Rietkerk_Model(
                    n=50,
                    plant_rate=plant_rate.value/10,
                    surface_rate=surface_rate.value/10,
                    infiltrated_rate=infiltrated_rate.value/10,
                    plant_loss_rate=plant_loss.value/100,
                    plant_water_uptake=plant_water_uptake.value/100,
                    k1=k1,
                    c=c,
                    plant_diffusion=plant_difussion.value/100,
                    infiltrated_diffusion=infiltrated_diffusion.value/100,
                    infiltrated_water_loss=infiltrated_water_loss.value/100,
                    alpha=alpha,
                    Wo=Wo,
                    k2=k2,
                    rainfall=rainfall.value,
                    surface_diffusion=water_surface_diffusion.value,
                ),
                [
                    plant_rate.value, surface_rate.value,infiltrated_rate.value,plant_loss.value, plant_water_uptake.value, plant_difussion.value,
                    infiltrated_diffusion.value, infiltrated_water_loss.value,
                    rainfall.value, water_surface_diffusion.value, trigger.value
                ]
            )

            step, set_step = solara.use_state(0)
            data, set_data = solara.use_state(model.export())
            is_running, set_is_running = solara.use_state(False)

            def reset_trigger():
                trigger.value += 1

            def reset_simulation():
                set_is_running(False)
                set_step(0)
                set_data(model.export())
            solara.use_effect(reset_simulation, [model])

            def run_simulation(cancel: threading.Event):
                if not is_running:
                    return

                current_step = step
                while not cancel.is_set():
                    for _ in range(100):
                        model.step()

                    current_step += 100

                    set_step(current_step)
                    set_data(model.export())

                    time.sleep(0.1)

            solara.use_thread(run_simulation, dependencies=[is_running, model])

            with solara.Column(style={"max-width": "80vw"}):
                with solara.Row(justify="center"):
                    fig, ax = plt.subplots(figsize=(7, 6))
                    im = ax.imshow(data, cmap='viridis', origin='lower')
                    ax.set_title(f'Passo atual: {step}')
                    ax.set_xlabel('x')
                    ax.set_ylabel('y')
                    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                    solara.FigureMatplotlib(fig)
                    plt.close(fig)
                with solara.Row(justify="center"):
                    solara.Button(
                        label="Parar" if is_running else "Iniciar",
                        on_click=lambda: set_is_running(not is_running),
                        color="error" if is_running else "success"
                    )
                    solara.Button("Dar passo", on_click=lambda: (model.step(), set_step(step + 1), set_data(model.export())), disabled=is_running)
                    solara.Button(label="Resetar", on_click=reset_trigger, disabled=is_running or step == 0, color="info")